# 描述统计、相关性与第一版基准诊断

Python 生成样本流、描述统计和 pairwise correlation；固定效应模型由 Stata 内置 xtreg 实际运行。当前结果是模拟训练数据的 baseline diagnostic，不是政策因果结论。

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, '..')
import pandas as pd

processed_dir = Path('../data/processed')
results_dir = Path('../results/first_stage')
results_dir.mkdir(parents=True, exist_ok=True)
variables = pd.read_parquet(processed_dir / 'research_panel_variables.parquet')
financial_raw = pd.read_excel('../data/raw/firm_financials.xlsx')
financial_clean = pd.read_parquet(processed_dir / 'firm_financials_clean.parquet')
profile = pd.read_parquet(processed_dir / 'firm_profile_clean.parquet')
patents_raw = pd.read_csv('../data/raw/patents.csv', dtype=object, keep_default_na=False)
patents = pd.read_parquet(processed_dir / 'patents_clean.parquet')
variables.head()

,stock_code,company_name,year,total_assets,total_liabilities,revenue,net_profit,cash,rd_expense,roe,...,rd_intensity,cash_ratio,employee_ln,firm_age,patent_total,patent_total_ln,invention_ln,citation_ln,invention_share,citations_per_invention
0,000001,华辰科技股份有限公司,2020,44813001602.0,23908847993.0,8788852319.0,823332467.0,4322096458.0,655221236.0,0.0394,...,0.074551,0.096447,9.597302,16.0,27.0,3.332205,2.197225,3.526361,0.296296,4.125
1,000001,华辰科技股份有限公司,2021,51047456089.0,30247752226.0,12258996387.0,1406421903.0,12041496671.0,680417512.0,0.0676,...,0.055504,0.235888,9.825202,17.0,24.0,3.218876,2.197225,3.178054,0.333333,2.875
2,000001,华辰科技有限公司,2022,53781621074.0,40472674936.0,11646604275.0,-90319782.0,8762245930.0,883046862.0,-0.0068,...,0.075820,0.162923,9.733292,18.0,NaN,NaN,NaN,NaN,NaN,NaN
3,000001,华辰科技股份有限公司,2023,1934677743.0,476851735.0,17627268570.0,-908582885.0,273990284.0,393474856.0,-0.6232,...,0.022322,0.141621,7.977282,19.0,17.0,2.890372,2.197225,3.218876,0.470588,3.000
4,000001,ST华辰科技股份有限公司,2024,48924685296.0,37663675836.0,35182016317.0,2123243663.0,13679245844.0,2724202038.0,0.1885,...,0.077432,0.279598,9.661225,20.0,18.0,2.944439,2.397895,3.401197,0.555556,2.900


In [2]:
flow = pd.DataFrame([
    {'stage': 'raw financial rows', 'value': len(financial_raw)},
    {'stage': 'clean financial firm-year', 'value': len(financial_clean)},
    {'stage': 'financial firms', 'value': variables['stock_code'].nunique()},
    {'stage': 'profile firms', 'value': profile['stock_code'].nunique()},
    {'stage': 'financial firms matched to profile', 'value': variables['stock_code'].nunique()},
    {'stage': 'raw patent rows', 'value': len(patents_raw)},
    {'stage': 'clean patent firm-year', 'value': len(patents)},
    {
        'stage': 'patent-observed research firm-year',
        'value': int(variables['patent_record_present'].sum()),
    },
    {'stage': 'final research panel firm-year', 'value': len(variables)},
])
flow.to_csv(results_dir / 'sample_flow.csv', index=False)
flow

,stage,value
0,raw financial rows,260
1,clean financial firm-year,240
2,financial firms,40
3,profile firms,42
4,financial firms matched to profile,40
5,raw patent rows,225
6,clean patent firm-year,220
7,patent-observed research firm-year,220
8,final research panel firm-year,240


In [3]:
descriptive_columns = [
    'total_assets', 'total_liabilities', 'revenue', 'net_profit', 'rd_expense',
    'roe', 'employees', 'size_ln', 'leverage', 'roa', 'rd_intensity',
    'cash_ratio', 'employee_ln', 'firm_age', 'invention_patents',
    'utility_patents', 'patent_citations', 'patent_total', 'patent_total_ln',
    'invention_ln', 'citation_ln', 'invention_share',
]
rows = []
for column in descriptive_columns:
    values = pd.to_numeric(variables[column], errors='coerce')
    rows.append({
        'variable': column,
        'N': int(values.notna().sum()),
        'missing': int(values.isna().sum()),
        'mean': values.mean(), 'std': values.std(),
        'min': values.min(), 'p25': values.quantile(.25),
        'median': values.median(), 'p75': values.quantile(.75),
        'max': values.max(),
    })
descriptive = pd.DataFrame(rows)
descriptive.to_csv(results_dir / 'descriptive_statistics.csv', index=False)
descriptive

,variable,N,missing,mean,std,min,p25,median,p75,max
0,total_assets,239,1,3.051901e+10,1.648720e+10,8.254876e+08,1.629561e+10,3.047498e+10,4.456676e+10,5.961285e+10
1,total_liabilities,239,1,1.426207e+10,1.029579e+10,4.403734e+08,6.583294e+09,1.179975e+10,2.030272e+10,4.657980e+10
2,revenue,240,0,2.301985e+10,1.380526e+10,4.167864e+08,1.032913e+10,2.375099e+10,3.562058e+10,4.478777e+10
3,net_profit,238,2,1.558869e+09,3.166962e+09,-9.949049e+09,3.095304e+07,8.316699e+08,3.043108e+09,1.471324e+10
4,rd_expense,239,1,1.366512e+09,1.154295e+09,1.799965e+07,4.203240e+08,1.078463e+09,2.069234e+09,4.982016e+09
5,roe,240,0,3.390500e-01,2.570382e+00,-1.916900e+00,1.300000e-03,5.245000e-02,2.391250e-01,3.859300e+01
6,employees,238,2,2.192668e+04,1.317598e+04,2.460000e+02,1.068475e+04,2.064450e+04,3.386475e+04,4.496100e+04
7,size_ln,239,1,2.391133e+01,8.025569e-01,2.053148e+01,2.351392e+01,2.414017e+01,2.452025e+01,2.481114e+01
8,leverage,238,2,4.657863e-01,1.839709e-01,1.567795e-01,3.027442e-01,4.658516e-01,6.264552e-01,7.986167e-01
9,roa,237,3,1.391273e-01,6.931815e-01,-1.136998e+00,1.072181e-03,2.860364e-02,1.208667e-01,8.951074e+00


In [4]:
correlation_columns = [
    'size_ln', 'leverage', 'roa', 'rd_intensity', 'cash_ratio',
    'employee_ln', 'patent_total_ln', 'invention_ln',
    'citation_ln', 'invention_share',
]
numeric = variables[correlation_columns].apply(pd.to_numeric, errors='coerce')
correlation_matrix = numeric.corr(method='pearson')
correlation_n = pd.DataFrame(index=correlation_columns, columns=correlation_columns, dtype='Int64')
for left in correlation_columns:
    for right in correlation_columns:
        correlation_n.loc[left, right] = int(numeric[[left, right]].dropna().shape[0])
correlation_matrix.to_csv(results_dir / 'correlation_matrix.csv')
correlation_n.to_csv(results_dir / 'correlation_n.csv')
correlation_matrix

,size_ln,leverage,roa,rd_intensity,cash_ratio,employee_ln,patent_total_ln,invention_ln,citation_ln,invention_share
size_ln,1.000000,-0.011559,-0.392018,0.111139,-0.056131,0.055817,-0.068019,-0.029994,-0.033963,0.026297
leverage,-0.011559,1.000000,0.022588,-0.029213,-0.053820,-0.006236,0.134913,0.132685,0.109926,0.072016
roa,-0.392018,0.022588,1.000000,0.053281,0.079297,0.037347,0.115378,0.041718,0.000805,-0.048955
rd_intensity,0.111139,-0.029213,0.053281,1.000000,0.092893,-0.043532,-0.011277,0.017961,0.008732,-0.002701
cash_ratio,-0.056131,-0.053820,0.079297,0.092893,1.000000,-0.040940,-0.025682,-0.023203,0.011737,0.001289
employee_ln,0.055817,-0.006236,0.037347,-0.043532,-0.040940,1.000000,0.059698,0.010199,0.005078,-0.039284
patent_total_ln,-0.068019,0.134913,0.115378,-0.011277,-0.025682,0.059698,1.000000,0.598213,0.519502,-0.059208
invention_ln,-0.029994,0.132685,0.041718,0.017961,-0.023203,0.010199,0.598213,1.000000,0.886898,0.751348
citation_ln,-0.033963,0.109926,0.000805,0.008732,0.011737,0.005078,0.519502,0.886898,1.000000,0.678279
invention_share,0.026297,0.072016,-0.048955,-0.002701,0.001289,-0.039284,-0.059208,0.751348,0.678279,1.000000


In [5]:
regression_path = results_dir / 'regression_results.csv'
if regression_path.exists():
    regression_results = pd.read_csv(regression_path)
    regression_results
else:
    print('先运行 stata/06_first_stage_analysis.do，再回读 regression_results.csv。')

## 解释边界

相关系数只表示 pairwise available observations 下的线性相关，不表示因果。三组 Stata 模型使用企业固定效应、年份固定效应和企业聚类标准误，仅用于验证变量和面板回归管线；没有政策变量，因此不能据此宣称政策连续性假设成立。